## 🎯 Learning Objectives
* Understand the critical role of external tools (APIs) in enabling multi-agent systems to perform real-world actions.
* Identify and categorize essential tools for a hotel reservation system: booking APIs, calendar APIs, and confirmation email services.
* Grasp the conceptual interface and expected functionality of each tool type from an agent's perspective.
* Recognize the importance of robust API integration, error handling, and data parsing for effective agent-tool interaction.


## Lesson PRJ01-L02: Tool Inventory: Booking APIs, Calendar, Confirmation Emails

Welcome to the "Project Scoping" phase of building your Multi-Agent AI System for Hotel Reservations! Just as a skilled artisan needs a well-stocked toolbox, an intelligent agent system requires a robust inventory of tools to interact with the real world. For our hotel reservation system, these tools are primarily **APIs (Application Programming Interfaces)** that allow our agents to perform actions like searching for hotels, making bookings, managing calendars, and sending/receiving emails.

Think of our multi-agent system as a highly efficient, digital travel agency. A human travel agent doesn't just *think* about booking a hotel; they *use* tools: a computer, booking websites, a calendar, and an email client. Our AI agents need their digital equivalents.

In 2026, the landscape of API integration is more sophisticated than ever. We're moving beyond simple REST calls to intelligent API discovery, AI-driven function calling, and highly standardized API descriptions (like OpenAPI 3.1). Our agents won't just *know* about tools; they'll be able to *reason* about which tool to use and *how* to use it based on their current goal.

Let's break down the core tools our hotel reservation system will need:

### 1. Booking APIs
These are the backbone of our system. They allow agents to interact directly with hotel booking platforms. Imagine an agent needing to find a hotel in Paris for specific dates. It doesn't *know* about all hotels; it queries a Booking API. These APIs typically offer functionalities such as:
*   **Searching for hotels:** Based on location, dates, number of guests, price range, amenities, etc.
*   **Retrieving hotel details:** Information about rooms, facilities, reviews, images.
*   **Checking room availability:** Real-time availability for specific room types and dates.
*   **Making reservations:** Submitting guest details and payment information to secure a booking.
*   **Modifying/Cancelling reservations:** Updating booking details or canceling a confirmed stay.

**Analogy:** This is like the travel agent using their specialized booking software or directly accessing a hotel chain's reservation system.

### 2. Calendar APIs
Once a booking is made, it's crucial to add it to a calendar. This ensures the guest (and potentially other agents in the system) has a clear record of their travel plans. Calendar APIs enable our agents to:
*   **Check availability:** See if specific time slots are free (though less critical for hotel bookings, it's vital for related tasks like flight scheduling or meeting coordination).
*   **Create events:** Add new calendar entries with details like hotel name, check-in/check-out dates, confirmation numbers, and location.
*   **Update/Delete events:** Modify or remove existing calendar entries.

**Analogy:** This is the travel agent adding the booking details to the client's digital calendar (e.g., Google Calendar, Outlook Calendar).

### 3. Confirmation Email Services
After a booking, a confirmation email is the primary way to communicate details to the user and receive official documentation. Our agents need to be able to both *send* and potentially *parse* these emails.
*   **Sending Confirmation Emails:** Composing and dispatching emails with booking summaries, terms and conditions, and other relevant information.
*   **Parsing Confirmation Emails (for verification/updates):** In some advanced scenarios, an agent might need to read an incoming email (e.g., from the hotel directly) to extract a confirmation number, check-in instructions, or detect changes.

**Analogy:** This is the travel agent sending a detailed itinerary to the client and also monitoring their inbox for any updates from the hotel.

### The Agentic Perspective
From an agent's viewpoint, these APIs are simply *functions* it can call. The agent doesn't care about the underlying HTTP requests or JSON parsing; it just needs to know: "What can this tool do?" and "What inputs does it need?" and "What output will I get?". Our goal in this project is to define these interfaces clearly so our agents can effectively utilize them to achieve the overarching goal of seamless hotel reservations.


In [ ]:
import datetime
import uuid
import time

# --- Mock Booking API --- 
# In a real system, this would interact with external services like Expedia, Booking.com, or direct hotel APIs.
class MockBookingAPI:
    def __init__(self):
        self.bookings = {}
        self.available_hotels = {
            "Paris": [
                {"id": "H001", "name": "Grand Hotel Paris", "price_per_night": 250, "currency": "EUR", "rooms_available": 10},
                {"id": "H002", "name": "Boutique Stay Paris", "price_per_night": 180, "currency": "EUR", "rooms_available": 5}
            ],
            "London": [
                {"id": "H003", "name": "The Londoner", "price_per_night": 300, "currency": "GBP", "rooms_available": 15},
                {"id": "H004", "name": "City View Inn", "price_per_night": 120, "currency": "GBP", "rooms_available": 20}
            ]
        }

    def search_hotels(self, location: str, check_in_date: str, check_out_date: str, guests: int = 1, max_price: float = None) -> list:
        """Simulates searching for hotels based on criteria."""
        print(f"[BookingAPI] Searching for hotels in {location} from {check_in_date} to {check_out_date} for {guests} guests...")
        time.sleep(1) # Simulate network latency
        
        results = []
        if location in self.available_hotels:
            for hotel in self.available_hotels[location]:
                if hotel["rooms_available"] > 0 and (max_price is None or hotel["price_per_night"] <= max_price):
                    # In a real API, availability would be dynamic per date range
                    results.append({
                        "hotel_id": hotel["id"],
                        "name": hotel["name"],
                        "price_per_night": hotel["price_per_night"],
                        "currency": hotel["currency"],
                        "availability_status": "available"
                    })
        return results

    def book_room(self, hotel_id: str, check_in_date: str, check_out_date: str, guests: int, guest_name: str, payment_info: dict) -> dict:
        """Simulates booking a room. Returns a confirmation."""
        print(f"[BookingAPI] Attempting to book hotel {hotel_id} for {guest_name}...")
        time.sleep(2) # Simulate booking processing time
        
        # Find the hotel across all locations
        found_hotel = None
        for loc_hotels in self.available_hotels.values():
            for hotel in loc_hotels:
                if hotel["id"] == hotel_id:
                    found_hotel = hotel
                    break
            if found_hotel: break

        if not found_hotel or found_hotel["rooms_available"] == 0:
            return {"status": "failed", "message": "Hotel not found or no rooms available."}

        booking_id = str(uuid.uuid4())
        confirmation = {
            "booking_id": booking_id,
            "hotel_id": hotel_id,
            "hotel_name": found_hotel["name"],
            "guest_name": guest_name,
            "check_in": check_in_date,
            "check_out": check_out_date,
            "guests": guests,
            "total_price": found_hotel["price_per_night"] * (datetime.datetime.strptime(check_out_date, '%Y-%m-%d') - datetime.datetime.strptime(check_in_date, '%Y-%m-%d')).days,
            "currency": found_hotel["currency"],
            "status": "confirmed"
        }
        self.bookings[booking_id] = confirmation
        found_hotel["rooms_available"] -= 1 # Decrement available rooms
        print(f"[BookingAPI] Booking {booking_id} confirmed for {guest_name} at {found_hotel['name']}.")
        return confirmation

    def cancel_booking(self, booking_id: str) -> dict:
        """Simulates canceling a booking."""
        print(f"[BookingAPI] Attempting to cancel booking {booking_id}...")
        time.sleep(1)
        if booking_id in self.bookings:
            booking = self.bookings.pop(booking_id)
            # Increment room availability (simplified)
            for loc_hotels in self.available_hotels.values():
                for hotel in loc_hotels:
                    if hotel["id"] == booking["hotel_id"]:
                        hotel["rooms_available"] += 1
                        break
            print(f"[BookingAPI] Booking {booking_id} cancelled.")
            return {"status": "cancelled", "booking_id": booking_id, "message": "Booking successfully cancelled."}
        return {"status": "failed", "message": "Booking ID not found."}

# --- Mock Calendar API --- 
# In a real system, this would interact with Google Calendar API, Outlook Calendar API, etc.
class MockCalendarAPI:
    def __init__(self):
        self.events = []

    def add_event(self, title: str, start_time: str, end_time: str, description: str = "", location: str = "") -> dict:
        """Simulates adding an event to a calendar."""
        print(f"[CalendarAPI] Adding event: '{title}' from {start_time} to {end_time}...")
        time.sleep(0.5)
        event_id = str(uuid.uuid4())
        event = {
            "event_id": event_id,
            "title": title,
            "start_time": start_time,
            "end_time": end_time,
            "description": description,
            "location": location,
            "status": "created"
        }
        self.events.append(event)
        print(f"[CalendarAPI] Event '{title}' added with ID: {event_id}.")
        return event

    def get_events(self, start_date: str, end_date: str) -> list:
        """Simulates retrieving events within a date range."""
        print(f"[CalendarAPI] Retrieving events from {start_date} to {end_date}...")
        time.sleep(0.5)
        # Simplified: In a real system, this would filter by date
        return [e for e in self.events if e["start_time"] >= start_date and e["end_time"] <= end_date]

# --- Mock Email Service --- 
# In a real system, this would interact with SendGrid, Mailgun, AWS SES, or a custom email server.
class MockEmailService:
    def __init__(self):
        self.sent_emails = []
        self.inbox = [] # For simulating incoming emails

    def send_email(self, recipient: str, subject: str, body: str, sender: str = "noreply@agenticlabs.ng") -> dict:
        """Simulates sending an email."""
        print(f"[EmailService] Sending email to {recipient} with subject: '{subject}'...")
        time.sleep(0.7)
        email_id = str(uuid.uuid4())
        email = {
            "email_id": email_id,
            "sender": sender,
            "recipient": recipient,
            "subject": subject,
            "body": body,
            "timestamp": datetime.datetime.now().isoformat(),
            "status": "sent"
        }
        self.sent_emails.append(email)
        print(f"[EmailService] Email sent to {recipient} (ID: {email_id}).")
        return email

    def receive_email(self, recipient: str) -> list:
        """Simulates receiving emails for a given recipient. (Simplified: returns all in inbox for now)"""
        print(f"[EmailService] Checking inbox for {recipient}...")
        time.sleep(0.5)
        # In a real system, this would fetch from an actual inbox and filter by recipient
        return [e for e in self.inbox if e["recipient"] == recipient]

    def simulate_incoming_confirmation(self, recipient: str, booking_details: dict):
        """Helper to simulate an incoming confirmation email from a hotel."""
        subject = f"Booking Confirmation for {booking_details['hotel_name']}"
        body = f"Dear {booking_details['guest_name']}, your booking (ID: {booking_details['booking_id']}) at {booking_details['hotel_name']} from {booking_details['check_in']} to {booking_details['check_out']} is confirmed. Total: {booking_details['total_price']} {booking_details['currency']}."
        self.inbox.append({
            "email_id": str(uuid.uuid4()),
            "sender": "hotel@example.com",
            "recipient": recipient,
            "subject": subject,
            "body": body,
            "timestamp": datetime.datetime.now().isoformat(),
            "status": "received"
        })
        print(f"[EmailService] Simulated incoming confirmation email for {recipient}.")

    def parse_confirmation_email(self, email_body: str) -> dict:
        """Simulates parsing an email body to extract booking details using NLP/regex (simplified)."""
        print("[EmailService] Parsing confirmation email...")
        time.sleep(0.3)
        # In 2026, this would likely involve an LLM or a sophisticated NLP model
        # For this mock, we'll just look for keywords
        parsed_data = {}
        if "Booking Confirmation" in email_body:
            parsed_data["type"] = "booking_confirmation"
            if "ID: " in email_body:
                parsed_data["booking_id"] = email_body.split("ID: ")[1].split(")")[0]
            if "at " in email_body:
                parsed_data["hotel_name"] = email_body.split("at ")[1].split(" from")[0]
            # ... more sophisticated parsing would go here
        return parsed_data

# --- Demonstrate Agent-Tool Interaction --- 

# Initialize our mock tools
booking_api = MockBookingAPI()
calendar_api = MockCalendarAPI()
email_service = MockEmailService()

print("\n--- Scenario: Agent searches, books, adds to calendar, and sends confirmation ---\n")

# 1. Agent searches for hotels
search_results = booking_api.search_hotels("Paris", "2026-07-10", "2026-07-15", guests=2, max_price=200)
print(f"Search Results: {search_results}\n")

if search_results:
    chosen_hotel = search_results[0] # Agent chooses the first available hotel
    print(f"Agent chose: {chosen_hotel['name']}\n")

    # 2. Agent books a room
    guest_name = "Alice Smith"
    payment_info = {"card_type": "Visa", "last_four": "1234"}
    booking_confirmation = booking_api.book_room(
        chosen_hotel["hotel_id"],
        "2026-07-10", "2026-07-15", 2, guest_name, payment_info
    )
    print(f"Booking Confirmation: {booking_confirmation}\n")

    if booking_confirmation["status"] == "confirmed":
        # 3. Agent adds to calendar
        event_title = f"Hotel Stay: {booking_confirmation['hotel_name']}"
        event_description = f"Booking ID: {booking_confirmation['booking_id']}. Check-in: {booking_confirmation['check_in']}, Check-out: {booking_confirmation['check_out']}."
        calendar_event = calendar_api.add_event(
            event_title,
            f"{booking_confirmation['check_in']}T15:00:00", # Standard check-in time
            f"{booking_confirmation['check_out']}T11:00:00", # Standard check-out time
            event_description,
            booking_confirmation['hotel_name']
        )
        print(f"Calendar Event: {calendar_event}\n")

        # 4. Agent sends confirmation email to user
        email_body = f"Dear {guest_name},\n\nYour hotel booking at {booking_confirmation['hotel_name']} is confirmed!\nBooking ID: {booking_confirmation['booking_id']}\nCheck-in: {booking_confirmation['check_in']}\nCheck-out: {booking_confirmation['check_out']}\nTotal Price: {booking_confirmation['total_price']} {booking_confirmation['currency']}\n\nWe look forward to your stay!\nAgenticLabs Travel Team"
        email_service.send_email(recipient="alice.smith@example.com", subject="Your Hotel Booking Confirmation", body=email_body)
        
        # 5. (Optional) Simulate receiving an email and parsing it
        print("\n--- Scenario: Agent receives and parses an incoming confirmation ---\n")
        email_service.simulate_incoming_confirmation("alice.smith@example.com", booking_confirmation)
        incoming_emails = email_service.receive_email("alice.smith@example.com")
        if incoming_emails:
            parsed_data = email_service.parse_confirmation_email(incoming_emails[0]["body"])
            print(f"Parsed incoming email data: {parsed_data}\n")

        # 6. Agent cancels booking (demonstration of another tool call)
        print("\n--- Scenario: Agent cancels a booking ---\n")
        cancel_result = booking_api.cancel_booking(booking_confirmation["booking_id"])
        print(f"Cancellation Result: {cancel_result}\n")

else:
    print("No hotels found matching criteria.")


### Interpreting the Code Output and Practical Considerations

The code above provides a **mock implementation** of the core APIs our multi-agent system will interact with. It's crucial to understand that these are simplified representations, but they demonstrate the *interface* and *flow* of interaction that our agents will expect.

**What the Output Shows:**
*   **Sequential Tool Use:** You can observe how an agent, or a sequence of agents, would call these tools in a logical order: `search_hotels` -> `book_room` -> `add_event` -> `send_email`. This highlights the orchestration challenge in multi-agent systems.
*   **Input/Output Structure:** Each mock function takes specific parameters (e.g., `location`, `check_in_date`) and returns structured data (e.g., a list of hotels, a booking confirmation dictionary). This structured data is what other agents or subsequent tool calls will consume.
*   **Simulated Latency:** The `time.sleep()` calls are important. They remind us that real-world API calls are not instantaneous. This latency has significant implications for agent design, especially when dealing with long-running tasks or needing quick responses.
*   **Error Handling (Implicit):** While our mock is basic, a real API would return detailed error codes and messages (e.g., `404 Not Found`, `400 Bad Request`, `500 Internal Server Error`). Agents must be designed to interpret and react to these errors, perhaps by retrying, escalating, or trying an alternative tool.

**Performance Trade-offs and Typical Use Cases:**

1.  **API Latency and Rate Limits:**
    *   **Trade-off:** Frequent API calls can lead to higher latency and hit rate limits imposed by providers. This can slow down the agent system or even lead to temporary bans.
    *   **Mitigation:** Agents should implement caching mechanisms for frequently accessed static data (e.g., hotel details that don't change often). For dynamic data, intelligent throttling, exponential backoff for retries, and parallel processing (where appropriate) are essential.
    *   **Use Case:** A `SearchAgent` might query a booking API, cache results for a short period, and then pass them to a `SelectionAgent`.

2.  **Data Consistency and Idempotency:**
    *   **Trade-off:** Network issues can lead to duplicate requests. Booking an item twice or adding the same calendar event multiple times is undesirable.
    *   **Mitigation:** APIs should ideally be *idempotent* for actions like booking (meaning calling it multiple times with the same parameters has the same effect as calling it once). Agents should also track the state of their actions to avoid redundant operations.
    *   **Use Case:** After a `BookingAgent` calls `book_room`, it should store the `booking_id` to prevent accidental re-booking and to enable subsequent actions like cancellation.

3.  **Security and Authentication:**
    *   **Trade-off:** Real APIs require authentication (API keys, OAuth tokens). Managing these securely is paramount.
    *   **Mitigation:** Credentials should never be hardcoded. Use environment variables, secure vaults, or dedicated credential management services. Agents must be designed to refresh tokens when they expire.
    *   **Use Case:** An `AuthAgent` or a shared `CredentialManager` module would handle obtaining and refreshing API tokens for all other agents.

4.  **Parsing and Data Transformation:**
    *   **Trade-off:** API responses can vary in format and require transformation to be useful for different agents or downstream systems.
    *   **Mitigation:** Implement robust data validation and transformation layers. For email parsing, as shown in the mock, advanced NLP techniques (like those powered by LLMs in 2026) will be crucial for extracting structured information from unstructured text.
    *   **Use Case:** A `ParserAgent` might specialize in taking raw API responses or email bodies and converting them into a standardized internal data format for the system.

By understanding these mock interfaces and the practical considerations, we lay the groundwork for designing robust, efficient, and intelligent agents that can effectively leverage external tools to achieve complex goals like managing hotel reservations.


### Resources

*   **OpenAPI Specification (Swagger):** The industry standard for defining RESTful APIs. Understanding this helps in designing and consuming APIs effectively. [OpenAPI Initiative](https://www.openapis.org/)
*   **Google Calendar API Documentation:** A real-world example of a robust calendar API. [Google Calendar API](https://developers.google.com/calendar/api/guides/overview)
*   **Microsoft Graph API (for Outlook Calendar):** Another major calendar and productivity API. [Microsoft Graph Calendar API](https://learn.microsoft.com/en-us/graph/api/resources/calendar?view=graph-rest-1.0)
*   **SendGrid Documentation:** A popular email sending service with comprehensive API documentation. [SendGrid API Docs](https://docs.sendgrid.com/api-reference/)
*   **Function Calling with LLMs (e.g., OpenAI, Google Gemini):** Explore how modern LLMs can be prompted to *choose* and *call* external tools based on user intent. This is a core concept for agentic systems in 2026. 
    *   [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
    *   [Google Gemini Function Calling](https://ai.google.dev/docs/function_calling)
*   **LangChain Tools and Agents:** A framework that simplifies integrating LLMs with external tools. [LangChain Tools](https://python.langchain.com/docs/modules/agents/tools/)
*   **CrewAI Tools:** Another popular framework for multi-agent systems, emphasizing tool integration. [CrewAI Tools](https://docs.crewai.com/how-to/Tools/)
